# Phase 5 — Layer-Specific SMI Effects (DREADD saline/DCZ cohort)

Track B only for now, same reasoning as Phase 4 — Track A's layer-specific
mixed-effects analysis is deferred until Track A gets revisited across
Phases 3–6 after this phase and Phase 6 are done for Track B.

Tests the deep-layer sensitivity hypothesis: since RSC projects directly
to V1's deep layers (L5/L6), DCZ's effect on spatial coding (if any) is
predicted to be stronger in L5/L6 than in superficial layers (L2/3, L4).

Loads Phase 4's already-saved comparison-group tables directly
(`{group}_comparison_table.csv`, from Function 4.8) rather than
re-deriving `all_group_dfs` via the interactive picker — the whole point
of saving those tables was so later phases wouldn't need to redo that.
Then adds layer-stratified and depth-interaction tests on top.

Built incrementally, one function at a time. Consolidated into
`5.LayerSpecific.py` only once everything here works end-to-end on real
data.

In [ ]:
import sys
sys.path.insert(0, r"C:\Users\jasmineyeo\Documents\GitHub\V1_SpatialModulation")

import os
import glob
import json
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Qt5Agg')
import matplotlib.pyplot as plt
from matplotlib import rcParams
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from scipy.stats import kruskal, mannwhitneyu, rankdata, ttest_rel, wilcoxon


# rcParams['legend.fontsize'] = 40
# rcParams['axes.labelsize'] = 40
# rcParams['axes.titlesize'] = 50
# rcParams['xtick.labelsize'] = 40
# rcParams['ytick.labelsize'] = 40

rcParams['legend.fontsize'] = 20
rcParams['axes.labelsize'] = 20
rcParams['axes.titlesize'] = 25
rcParams['xtick.labelsize'] = 20
rcParams['ytick.labelsize'] = 20

# One real DREADD animal's already-saved Phase 4 output, to develop and
# sanity-check against.
TEST_ANIMAL_DIR = r"D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY093_V1prism_DREADD"

## Setup — load Phase 4's already-saved comparison tables

Phase 4's Function 4.8 already saved every comparison group's per-cell
table to `{group_name}_comparison_table.csv` under
`{ANIMAL_DIR}/Phase4_SessionComparison_Results/`. This just loads all of
them back into the same `{group_name: df}` shape `all_group_dfs` always
had — no interactive picking needed.

- **Input:** `output_dir` (the animal's `Phase4_SessionComparison_Results`
  folder).
- **Output:** `all_group_dfs` (`{group_name: df}`).

In [52]:
def load_all_group_dfs_from_phase4(output_dir):
    """
    Load every comparison group's table already saved by Phase 4's
    Function 4.8 (save_all_phase4_outputs). See markdown above.

    Parameters
    ----------
    output_dir : str
        e.g. os.path.join(ANIMAL_DIR, 'Phase4_SessionComparison_Results').

    Returns
    -------
    all_group_dfs : dict
        {group_name: df}.
    """
    csv_paths = sorted(glob.glob(os.path.join(output_dir, '*_comparison_table.csv')))

    if not csv_paths:
        raise FileNotFoundError(f"No *_comparison_table.csv found in {output_dir} -- "
                                 "has Phase 4's save step (Function 4.8) been run for this animal?")

    all_group_dfs = {}
    for csv_path in csv_paths:
        group_name = os.path.basename(csv_path)[:-len('_comparison_table.csv')]
        df = pd.read_csv(csv_path)
        all_group_dfs[group_name] = df
        print(f"Loaded '{group_name}': {len(df)} cell-rows <- {csv_path}")

    print(f"\nLoaded {len(all_group_dfs)} group(s) from {output_dir}: {list(all_group_dfs.keys())}")
    return all_group_dfs

In [53]:
# --- Load the comparison tables Phase 4 already saved for this animal ---
OUTPUT_DIR = os.path.join(TEST_ANIMAL_DIR, 'Phase4_SessionComparison_Results')

all_group_dfs = load_all_group_dfs_from_phase4(OUTPUT_DIR)

Loaded 'Active_OL': 2158 cell-rows <- D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD\Phase4_SessionComparison_Results\Active_OL_comparison_table.csv
Loaded 'DCZ1': 1261 cell-rows <- D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD\Phase4_SessionComparison_Results\DCZ1_comparison_table.csv
Loaded 'DCZ2': 1283 cell-rows <- D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD\Phase4_SessionComparison_Results\DCZ2_comparison_table.csv
Loaded 'DCZ3': 1290 cell-rows <- D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD\Phase4_SessionComparison_Results\DCZ3_comparison_table.csv
Loaded 'Stationary_OL': 1934 cell-rows <- D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD\Phase4_SessionComparison_Results\Stationary_OL_comparison_table.csv
Loaded 'baseline': 3331 cell-rows <- D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD\Phase4_SessionComparison_Results\baseline_comparison_table.csv

Loaded 6 group(s) fro

## Function 5.1 — `compare_smi_across_conditions`

Reimplemented from Phase 4's Function 4.7, unchanged — the core
Kruskal-Wallis omnibus + pairwise Mann-Whitney U (Holm-corrected within
the group's own pairwise set) logic, reused by every layer-level test
below.

- **Input:** `df` (one group's or one layer's table), `group_col`,
  `value_col`, `filter_col`.
- **Output:** `dict` or `None` (fewer than 2 categories present).

In [54]:
def compare_smi_across_conditions(df, group_col='condition', value_col='SMI', filter_col='valid'):
    """
    Kruskal-Wallis omnibus + pairwise Mann-Whitney U (Holm-corrected)
    across whatever categories are present. See markdown above.
    """
    filtered = df[df[filter_col]]
    categories = [c for c in filtered[group_col].unique() if pd.notna(c)]

    if len(categories) < 2:
        print(f"Only {len(categories)} category(ies) present after filtering on '{filter_col}' "
              f"-- nothing to compare ({categories}).")
        return None

    samples = {cat: filtered.loc[filtered[group_col] == cat, value_col].to_numpy()
               for cat in categories}

    group_medians = {cat: float(np.median(vals)) for cat, vals in samples.items()}
    group_n = {cat: len(vals) for cat, vals in samples.items()}

    omnibus_stat, omnibus_p = kruskal(*samples.values())

    pairwise_rows = []
    for cat_a, cat_b in combinations(categories, 2):
        u_stat, p_raw = mannwhitneyu(samples[cat_a], samples[cat_b], alternative='two-sided')
        pairwise_rows.append({
            'cond_a': cat_a, 'cond_b': cat_b,
            'median_diff': group_medians[cat_a] - group_medians[cat_b],
            'U_stat': u_stat, 'p_raw': p_raw,
        })

    pairwise_df = pd.DataFrame(pairwise_rows)
    if len(pairwise_df) > 0:
        _, p_holm, _, _ = multipletests(pairwise_df['p_raw'], method='holm')
        pairwise_df['p_holm'] = p_holm

    print(f"Categories ({group_col}): {categories}")
    print(f"  n per category: {group_n}")
    print(f"  median {value_col} per category: {group_medians}")
    print(f"  Kruskal-Wallis: H={omnibus_stat:.3f}, p={omnibus_p:.4f}")
    print(f"\n  Pairwise (Holm-corrected):")
    print(pairwise_df.to_string(index=False))

    return {
        'group_medians': group_medians,
        'group_n': group_n,
        'omnibus_stat': omnibus_stat,
        'omnibus_p': omnibus_p,
        'pairwise': pairwise_df,
    }

## Function 5.2 — `compare_smi_by_layer`

Loops Function 5.1 over each layer present (L2/3, L4, L5, L6, in that
canonical order where present) in one group's table, returning
`{layer: result}` plus a combined summary table (layer × pairwise
comparison × `p_holm`) for quick scanning of which layers show a
significant condition effect.

**Caveat**: each layer's Holm correction is applied only within that
layer's own ~3 pairwise tests (same as Function 4.7), not across all
layers × pairs combined for a group (e.g. 4 layers × 3 pairs = 12 tests).
The summary table doesn't correct for that broader multiplicity — keep
that in mind if scanning across layers for "any significant result."

- **Input:** `df` (one group's table).
- **Output:** `results` (`{layer: dict or None}`), `summary_df`.

In [55]:
CANONICAL_LAYER_ORDER = ['L2/3', 'L4', 'L5', 'L6']


def _layer_order(layers_present):
    return ([l for l in CANONICAL_LAYER_ORDER if l in layers_present]
            + [l for l in layers_present if l not in CANONICAL_LAYER_ORDER])


def compare_smi_by_layer(df, layer_col='layer', group_col='condition', value_col='SMI', filter_col='valid'):
    """
    Loop compare_smi_across_conditions over each layer present. See
    markdown above.

    Parameters
    ----------
    df : pandas.DataFrame
        One group's table (e.g. from build_comparison_table).
    layer_col, group_col, value_col, filter_col : str

    Returns
    -------
    results : dict
        {layer: compare_smi_across_conditions(...) result or None}.
    summary_df : pandas.DataFrame
        One row per (layer, pairwise comparison) that had 2+ categories.
    """
    layers_present = [l for l in df[layer_col].dropna().unique()]
    layer_order = _layer_order(layers_present)

    results = {}
    summary_rows = []
    for layer in layer_order:
        print(f"\n--- Layer {layer} ---")
        layer_df = df[df[layer_col] == layer]
        result = compare_smi_across_conditions(layer_df, group_col=group_col, value_col=value_col,
                                                filter_col=filter_col)
        results[layer] = result
        if result is not None:
            for _, row in result['pairwise'].iterrows():
                summary_rows.append({
                    'layer': layer, 'cond_a': row['cond_a'], 'cond_b': row['cond_b'],
                    'median_diff': row['median_diff'], 'p_holm': row['p_holm'],
                })

    summary_df = pd.DataFrame(summary_rows)
    if len(summary_df) > 0:
        print("\n=== Summary across layers (each layer's own Holm correction -- not corrected across layers) ===")
        print(summary_df.to_string(index=False))

    return results, summary_df

## Function 5.3 — `plot_smi_by_layer`

A grid (one panel per layer present) of the same violin+strip plot
Function 4.7 used, so all layers are visible side by side for one
comparison group.

- **Input:** `df` (one group's table).
- **Output:** `fig`.

In [56]:
def plot_smi_by_layer(df, layer_col='layer', group_col='condition', value_col='SMI', filter_col='valid', title=''):
    """
    Grid of violin+strip plots, one panel per layer. See markdown above.
    """
    filtered = df[df[filter_col]]
    layers_present = [l for l in filtered[layer_col].dropna().unique()]
    layer_order = _layer_order(layers_present)

    color_by_category = {'baseline': 'tab:blue', 'saline': 'tab:orange', 'dcz': 'tab:green'}

    n_layers = len(layer_order)
    n_cols = 2
    n_rows = int(np.ceil(n_layers / n_cols)) if n_layers > 0 else 1
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(8 * n_cols, 7 * n_rows))
    axes = np.atleast_1d(axes).flatten()

    for ax, layer in zip(axes, layer_order):
        layer_df = filtered[filtered[layer_col] == layer]
        category_order = [c for c in ['baseline', 'saline', 'dcz'] if c in layer_df[group_col].unique()]
        category_order += [c for c in layer_df[group_col].unique() if c not in category_order]

        data_by_category = [layer_df.loc[layer_df[group_col] == cat, value_col].to_numpy()
                            for cat in category_order]

        if len(data_by_category) == 0 or all(len(d) == 0 for d in data_by_category):
            ax.set_title(f"{layer} (no data)")
            ax.axis('off')
            continue

        parts = ax.violinplot(data_by_category, showmedians=True)
        for i, body in enumerate(parts['bodies']):
            body.set_facecolor(color_by_category.get(category_order[i], 'gray'))
            body.set_alpha(0.4)

        rng = np.random.default_rng(0)
        for i, vals in enumerate(data_by_category):
            jitter = rng.uniform(-0.08, 0.08, size=len(vals))
            ax.scatter(np.full(len(vals), i + 1) + jitter, vals,
                       color=color_by_category.get(category_order[i], 'gray'), s=12, alpha=0.5)

        ax.set_ylim(-1.3, 1.3)
        ax.set_yticks(np.arange(-1, 1.1, 0.5))
        ax.set_xticks(range(1, len(category_order) + 1))
        ax.set_xticklabels(category_order)
        ax.set_ylabel(value_col)
        ax.set_title(layer)
        ax.axhline(0, color='gray', linestyle='--', alpha=0.5)

    for ax in axes[len(layer_order):]:
        ax.axis('off')

    fig.suptitle(title, fontsize=20, fontweight='bold')
    plt.tight_layout()
    return fig

## Function 5.4 — `test_layer_depth_interaction`

The more rigorous test of the actual hypothesis: collapses layers into
`depth_group` (`L2/3`+`L4` = superficial, `L5`+`L6` = deep), rank-transforms
SMI (robust to non-normality, consistent with the non-parametric choice in
Function 4.7/5.1), and fits `rank(SMI) ~ C(condition) * C(depth_group)`.
The interaction term directly tests whether the condition effect *size*
differs between deep and superficial layers — this is the real
layer-specificity test, not just eyeballing whether per-layer p-values
from Function 5.2 happen to differ.

- **Input:** `df` (one group's table).
- **Output:** `model_result` (statsmodels OLS result) or `None`
  (fewer than 2 conditions or depth groups present).

In [57]:
def test_layer_depth_interaction(df, layer_col='layer', group_col='condition', value_col='SMI', filter_col='valid',
                                  deep_layers=('L5', 'L6'), superficial_layers=('L2/3', 'L4')):
    """
    rank(SMI) ~ C(condition) * C(depth_group). See markdown above.
    """
    filtered = df[df[filter_col]].copy()

    def _depth(layer):
        if layer in deep_layers:
            return 'deep'
        elif layer in superficial_layers:
            return 'superficial'
        return None

    filtered['depth_group'] = filtered[layer_col].map(_depth)
    filtered = filtered.dropna(subset=['depth_group', group_col, value_col])

    conditions_present = filtered[group_col].unique()
    depths_present = filtered['depth_group'].unique()

    if len(conditions_present) < 2 or len(depths_present) < 2:
        print(f"Not enough categories to test an interaction (conditions={list(conditions_present)}, "
              f"depths={list(depths_present)}) -- skipping.")
        return None

    filtered['SMI_rank'] = rankdata(filtered[value_col])

    formula = f"SMI_rank ~ C({group_col}) * C(depth_group)"
    model_result = smf.ols(formula, data=filtered).fit()

    print(f"\n=== Layer-depth interaction: rank({value_col}) ~ {group_col} * depth_group ===")
    print(model_result.summary().tables[1])

    interaction_terms = [p for p in model_result.params.index if ':' in p]
    if interaction_terms:
        print(f"\nInteraction term(s): {interaction_terms}")
        print("(a significant interaction term means the condition effect's SIZE differs "
              "between deep and superficial layers)")

    return model_result

## Function 5.7 — `compare_smi_by_depth_group`

Same core comparison as Function 5.2, but pooling layers into just two
buckets first (`deep` = L5+L6, `superficial` = L2/3+L4) rather than
keeping all four separate. More cells per bucket than any single layer,
so more power than the per-layer breakdown — a direct, easily-read
complement to Function 5.4's formal interaction p-value (which tests
whether the two differ from *each other*, not what each one's own
comparison looks like on its own).

- **Input:** `df` (one group's table).
- **Output:** `results` (`{'deep': dict or None, 'superficial': dict or None}`),
  `summary_df`.

In [58]:
def _assign_depth_group(layer_series, deep_layers=('L5', 'L6'), superficial_layers=('L2/3', 'L4')):
    """Map a 'layer' column to 'deep'/'superficial'/None. Shared by Functions 5.4/5.7/5.8/5.9."""
    def _depth(layer):
        if layer in deep_layers:
            return 'deep'
        elif layer in superficial_layers:
            return 'superficial'
        return None
    return layer_series.map(_depth)


def compare_smi_by_depth_group(df, layer_col='layer', group_col='condition', value_col='SMI', filter_col='valid',
                                deep_layers=('L5', 'L6'), superficial_layers=('L2/3', 'L4')):
    """
    Loop compare_smi_across_conditions over the two pooled depth groups
    ('deep' = L5+L6, 'superficial' = L2/3+L4) instead of all four layers.
    See markdown above.

    Parameters
    ----------
    df : pandas.DataFrame
        One group's table.
    layer_col, group_col, value_col, filter_col : str
    deep_layers, superficial_layers : tuple of str

    Returns
    -------
    results : dict
        {'deep': compare_smi_across_conditions(...) result or None,
         'superficial': ... }.
    summary_df : pandas.DataFrame
        One row per (depth_group, pairwise comparison) that had 2+ categories.
    """
    df = df.copy()
    df['depth_group'] = _assign_depth_group(df[layer_col], deep_layers, superficial_layers)

    results = {}
    summary_rows = []
    for depth_group, layers in (('deep', deep_layers), ('superficial', superficial_layers)):
        print(f"\n--- Depth group: {depth_group} ({'+'.join(layers)}) ---")
        depth_df = df[df['depth_group'] == depth_group]
        result = compare_smi_across_conditions(depth_df, group_col=group_col, value_col=value_col,
                                                filter_col=filter_col)
        results[depth_group] = result
        if result is not None:
            for _, row in result['pairwise'].iterrows():
                summary_rows.append({
                    'depth_group': depth_group, 'cond_a': row['cond_a'], 'cond_b': row['cond_b'],
                    'median_diff': row['median_diff'], 'p_holm': row['p_holm'],
                })

    summary_df = pd.DataFrame(summary_rows)
    if len(summary_df) > 0:
        print("\n=== Summary: deep vs superficial (each depth group's own Holm correction) ===")
        print(summary_df.to_string(index=False))

    return results, summary_df


## Function 5.8 — `plot_smi_by_depth_group`

Same violin+strip style as Function 5.3, but two panels (`deep`,
`superficial`) instead of four.

- **Input:** `df` (one group's table).
- **Output:** `fig`.

In [59]:
def plot_smi_by_depth_group(df, layer_col='layer', group_col='condition', value_col='SMI', filter_col='valid',
                             deep_layers=('L5', 'L6'), superficial_layers=('L2/3', 'L4'), title=''):
    """
    Violin+strip plots, one panel for 'deep' and one for 'superficial'.
    See markdown above.
    """
    filtered = df[df[filter_col]].copy()
    filtered['depth_group'] = _assign_depth_group(filtered[layer_col], deep_layers, superficial_layers)

    color_by_category = {'baseline': 'tab:blue', 'saline': 'tab:orange', 'dcz': 'tab:green'}

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    for ax, depth_group in zip(axes, ('deep', 'superficial')):
        depth_df = filtered[filtered['depth_group'] == depth_group]
        category_order = [c for c in ['baseline', 'saline', 'dcz'] if c in depth_df[group_col].unique()]
        category_order += [c for c in depth_df[group_col].unique() if c not in category_order]

        data_by_category = [depth_df.loc[depth_df[group_col] == cat, value_col].to_numpy()
                            for cat in category_order]

        if len(data_by_category) == 0 or all(len(d) == 0 for d in data_by_category):
            ax.set_title(f"{depth_group} (no data)")
            ax.axis('off')
            continue

        parts = ax.violinplot(data_by_category, showmedians=True)
        for i, body in enumerate(parts['bodies']):
            body.set_facecolor(color_by_category.get(category_order[i], 'gray'))
            body.set_alpha(0.4)

        rng = np.random.default_rng(0)
        for i, vals in enumerate(data_by_category):
            jitter = rng.uniform(-0.08, 0.08, size=len(vals))
            ax.scatter(np.full(len(vals), i + 1) + jitter, vals,
                       color=color_by_category.get(category_order[i], 'gray'), s=12, alpha=0.5)

        ax.set_xticks(range(1, len(category_order) + 1))
        ax.set_xticklabels(category_order)
        ax.set_ylabel(value_col)
        ax.set_title(depth_group)
        ax.axhline(0, color='gray', linestyle='--', alpha=0.5)

    fig.suptitle(title, fontsize=20, fontweight='bold')
    plt.tight_layout()
    return fig


## Function 5.9 — `summarize_layer_sample_sizes`

A diagnostic, not a comparison: tabulates n (valid cells) per layer ×
condition and per depth-group × condition, flagging anything below
`low_n_threshold`. We ran into saline n=3–6 in several individual layers
while reading Function 5.2's output earlier — that caveat was only
something said out loud, not something the notebook itself surfaced. This
makes it visible at a glance on every future run instead of relying on
someone noticing it in the printed pairwise tables.

- **Input:** `df` (one group's table), `low_n_threshold=10`.
- **Output:** `layer_counts`, `depth_counts` (both `pandas.DataFrame`,
  layers/depth-groups × conditions).

In [60]:
def summarize_layer_sample_sizes(df, layer_col='layer', group_col='condition', filter_col='valid',
                                  deep_layers=('L5', 'L6'), superficial_layers=('L2/3', 'L4'),
                                  low_n_threshold=10):
    """
    Tabulate n (valid cells) per layer x condition and per depth_group x
    condition, flagging anything below low_n_threshold. See markdown above.

    Parameters
    ----------
    df : pandas.DataFrame
        One group's table.
    layer_col, group_col, filter_col : str
    deep_layers, superficial_layers : tuple of str
    low_n_threshold : int

    Returns
    -------
    layer_counts : pandas.DataFrame
        One row per layer, one column per condition, n = valid cells.
    depth_counts : pandas.DataFrame
        Same, but for the two pooled depth groups.
    """
    filtered = df[df[filter_col]].copy()
    filtered['depth_group'] = _assign_depth_group(filtered[layer_col], deep_layers, superficial_layers)

    layers_present = [l for l in filtered[layer_col].dropna().unique()]
    layer_order = _layer_order(layers_present)
    layer_counts = filtered.groupby([layer_col, group_col]).size().unstack(fill_value=0).reindex(layer_order)

    depth_counts = filtered.groupby(['depth_group', group_col]).size().unstack(fill_value=0)
    depth_counts = depth_counts.reindex(['deep', 'superficial'])

    print("Sample sizes (valid cells) per layer x condition:")
    print(layer_counts.to_string())
    low_layer = layer_counts[layer_counts.lt(low_n_threshold).any(axis=1)]
    if len(low_layer) > 0:
        print(f"\nWARNING: layer(s) with a condition below n={low_n_threshold} "
              f"-- interpret those specific comparisons cautiously:")
        print(low_layer.to_string())

    print("\nSample sizes (valid cells) per depth group x condition:")
    print(depth_counts.to_string())
    low_depth = depth_counts[depth_counts.lt(low_n_threshold).any(axis=1)]
    if len(low_depth) > 0:
        print(f"\nWARNING: depth group(s) with a condition below n={low_n_threshold}:")
        print(low_depth.to_string())

    return layer_counts, depth_counts


## Functions 5.5/5.6 — drivers

1. **`run_layer_analysis_for_group`** — runs 5.9 (sample-size diagnostic,
   first) + 5.2 + 5.3 (4-layer breakdown) + 5.7 + 5.8 (deep/superficial
   pooled) + 5.4 (formal interaction test) for one group, printing/plotting
   everything together.
2. **`run_layer_analysis_all_groups`** — loops (1) over every group in
   `all_group_dfs`, skipping single-condition groups (e.g. `baseline`) --
   unlike Function 4.7, there's no session_label fallback here, since a
   depth-interaction test needs an actual condition contrast to test
   against depth; a day-of-baseline breakdown doesn't have one.

- **Input:** `all_group_dfs` (from the Setup cells above).
- **Output:** `{group_name: {'layer_counts', 'depth_counts', 'layer_results',
  'layer_summary', 'depth_results', 'depth_summary', 'interaction_result'}}`.

In [61]:
def run_layer_analysis_for_group(df, group_name=''):
    """
    Runs Functions 5.9 + 5.2 + 5.3 + 5.7 + 5.8 + 5.4 for one group. See
    markdown above. Retains both figures under 'layer_fig'/'depth_fig' so
    Function 5.10 can save them without needing to replot.
    """
    print(f"\n{'='*90}\nLayer analysis: {group_name}\n{'='*90}")

    layer_counts, depth_counts = summarize_layer_sample_sizes(df)

    layer_results, layer_summary_df = compare_smi_by_layer(df)
    layer_fig = plot_smi_by_layer(df, title=group_name)
    # plt.show()

    depth_results, depth_summary_df = compare_smi_by_depth_group(df)
    depth_fig = plot_smi_by_depth_group(df, title=f"{group_name} (deep vs superficial)")
    # plt.show()

    interaction_result = test_layer_depth_interaction(df)

    return {
        'layer_counts': layer_counts,
        'depth_counts': depth_counts,
        'layer_results': layer_results,
        'layer_summary': layer_summary_df,
        'layer_fig': layer_fig,
        'depth_results': depth_results,
        'depth_summary': depth_summary_df,
        'depth_fig': depth_fig,
        'interaction_result': interaction_result,
    }


def run_layer_analysis_all_groups(all_group_dfs, group_col='condition', filter_col='valid'):
    """
    Loops run_layer_analysis_for_group over every group, skipping
    single-condition groups (no condition contrast to test). See markdown
    above.
    """
    results = {}
    for group_name, df in all_group_dfs.items():
        conditions_present = df.loc[df[filter_col], group_col].unique()
        if len(conditions_present) < 2:
            print(f"\n{'='*90}\n{group_name}: only {len(conditions_present)} condition(s) present "
                  f"({list(conditions_present)}) -- skipping layer analysis for this group.\n{'='*90}")
            continue
        results[group_name] = run_layer_analysis_for_group(df, group_name=group_name)

    return results

In [62]:
# --- Run the layer-specific analysis across all your comparison groups ---
layer_analysis_results = run_layer_analysis_all_groups(all_group_dfs)


Layer analysis: Active_OL
Sample sizes (valid cells) per layer x condition:
condition  baseline  dcz  saline
layer                           
L2/3             31   26       8
L4               67   62      16
L5               98   90      39
L6               54   59      27

condition  baseline  dcz  saline
layer                           
L2/3             31   26       8

Sample sizes (valid cells) per depth group x condition:
condition    baseline  dcz  saline
depth_group                       
deep              152  149      66
superficial        98   88      24

--- Layer L2/3 ---


Categories (condition): ['baseline', 'dcz', 'saline']
  n per category: {'baseline': 31, 'dcz': 26, 'saline': 8}
  median SMI per category: {'baseline': 0.4776661457522118, 'dcz': 0.6779630646720183, 'saline': 0.8165311762980477}
  Kruskal-Wallis: H=11.394, p=0.0034

  Pairwise (Holm-corrected):
  cond_a cond_b  median_diff  U_stat    p_raw   p_holm
baseline    dcz    -0.200297   238.0 0.008400 0.016799
baseline saline    -0.338865    45.0 0.004642 0.013927
     dcz saline    -0.138568    76.0 0.270129 0.270129

--- Layer L4 ---
Categories (condition): ['baseline', 'dcz', 'saline']
  n per category: {'baseline': 67, 'dcz': 62, 'saline': 16}
  median SMI per category: {'baseline': 0.6144592000153275, 'dcz': 0.6431046402346888, 'saline': 0.5925083896828864}
  Kruskal-Wallis: H=1.355, p=0.5080

  Pairwise (Holm-corrected):
  cond_a cond_b  median_diff  U_stat    p_raw   p_holm
baseline    dcz    -0.028645  1831.0 0.247159 0.741478
baseline saline     0.021951   509.0 0.759670 1.000000
   

## Function 5.10 — save everything Phase 5 generates

Same reasoning as Phase 4's Function 4.8: nothing here was persisted to
disk until now — the sample-size tables, layer/depth stats, both figures
per group, and the interaction regression only ever lived in the kernel's
memory.

Saved under `{ANIMAL_DIR}/Phase5_LayerSpecific_Results/`:
- `{group}_layer_sample_sizes.csv` / `{group}_depth_sample_sizes.csv` —
  Function 5.9's tables.
- `{group}_layer_pairwise_stats.csv` / `{group}_depth_pairwise_stats.csv` —
  Functions 5.2/5.7's cross-layer/cross-depth summaries.
- `{group}_layer_omnibus_stats.json` / `{group}_depth_omnibus_stats.json` —
  per-layer/per-depth-group medians, n, and Kruskal-Wallis stat/p.
- `{group}_layer_comparison_plot.png` / `{group}_depth_comparison_plot.png` —
  Functions 5.3/5.8's figures.
- `{group}_interaction_regression.txt` — Function 5.4's full regression
  summary.

Built as small single-purpose pieces (reimplemented from Phase 4's
Function 4.8, since digit-prefixed module filenames can't be imported),
so any of them can be reused independently later:
1. **`save_dataframe_csv`** / **`save_figure_png`** / **`save_json`** —
   generic helpers, creating `output_dir` if needed.
2. **`_extract_omnibus_summary`** — pulls the medians/n/omnibus stat out
   of a `{category: compare_smi_across_conditions result}` dict for JSON
   saving.
3. **`save_layer_analysis_group_outputs`** — one group's full set of files.
4. **`save_all_layer_analysis_outputs`** — loops (3) over every group.

- **Input:** `output_dir`, `layer_analysis_results` (from Function 5.6,
  now carrying `'layer_fig'`/`'depth_fig'` keys per group).
- **Output:** `saved_paths_by_group` (nested dict of every path written).

In [63]:
def save_dataframe_csv(df, output_dir, filename, index=False):
    """
    Save a DataFrame to {output_dir}/{filename}, creating output_dir if
    needed. index=True for tables whose index is meaningful (e.g. layer
    names), False for tables with a plain range index.
    """
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, filename)
    df.to_csv(save_path, index=index)
    print(f"Saved -> {save_path}")
    return save_path


def save_figure_png(fig, output_dir, filename, dpi=150):
    """
    Save a matplotlib figure to {output_dir}/{filename}, creating
    output_dir if needed.
    """
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, filename)
    fig.savefig(save_path, dpi=dpi, bbox_inches='tight')
    print(f"Saved -> {save_path}")
    return save_path


def _json_safe(obj):
    """Recursively convert numpy scalar types to native Python for json.dump."""
    if isinstance(obj, dict):
        return {k: _json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_json_safe(v) for v in obj]
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    return obj


def save_json(data, output_dir, filename):
    """
    Save a JSON-serializable dict to {output_dir}/{filename}, creating
    output_dir if needed.
    """
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, filename)
    with open(save_path, 'w') as f:
        json.dump(_json_safe(data), f, indent=2)
    print(f"Saved -> {save_path}")
    return save_path


def _extract_omnibus_summary(results_by_category):
    """
    Pull {category: {'omnibus_stat', 'omnibus_p', 'group_medians', 'group_n'}}
    out of a {category: compare_smi_across_conditions result or None} dict,
    for JSON saving (skips categories where result was None).
    """
    summary = {}
    for cat, result in results_by_category.items():
        if result is None:
            continue
        summary[cat] = {
            'omnibus_stat': result['omnibus_stat'],
            'omnibus_p': result['omnibus_p'],
            'group_medians': result['group_medians'],
            'group_n': result['group_n'],
        }
    return summary


def save_layer_analysis_group_outputs(output_dir, group_name, result):
    """
    Save one group's layer-analysis outputs: sample-size tables, per-layer
    and per-depth-group summary stats (+ omnibus JSON), both figures, and
    the interaction regression's text summary. See markdown above.

    Parameters
    ----------
    output_dir : str
    group_name : str
    result : dict
        From run_layer_analysis_for_group (carries 'layer_fig'/'depth_fig').

    Returns
    -------
    saved_paths : dict
    """
    saved_paths = {
        'layer_sample_sizes': save_dataframe_csv(
            result['layer_counts'], output_dir, f"{group_name}_layer_sample_sizes.csv", index=True),
        'depth_sample_sizes': save_dataframe_csv(
            result['depth_counts'], output_dir, f"{group_name}_depth_sample_sizes.csv", index=True),
    }

    if len(result['layer_summary']) > 0:
        saved_paths['layer_pairwise'] = save_dataframe_csv(
            result['layer_summary'], output_dir, f"{group_name}_layer_pairwise_stats.csv")
    if len(result['depth_summary']) > 0:
        saved_paths['depth_pairwise'] = save_dataframe_csv(
            result['depth_summary'], output_dir, f"{group_name}_depth_pairwise_stats.csv")

    saved_paths['layer_omnibus'] = save_json(
        _extract_omnibus_summary(result['layer_results']), output_dir, f"{group_name}_layer_omnibus_stats.json")
    saved_paths['depth_omnibus'] = save_json(
        _extract_omnibus_summary(result['depth_results']), output_dir, f"{group_name}_depth_omnibus_stats.json")

    layer_fig = result.get('layer_fig')
    if layer_fig is not None:
        saved_paths['layer_plot'] = save_figure_png(layer_fig, output_dir, f"{group_name}_layer_comparison_plot.png")

    depth_fig = result.get('depth_fig')
    if depth_fig is not None:
        saved_paths['depth_plot'] = save_figure_png(depth_fig, output_dir, f"{group_name}_depth_comparison_plot.png")

    interaction_result = result.get('interaction_result')
    if interaction_result is not None:
        os.makedirs(output_dir, exist_ok=True)
        txt_path = os.path.join(output_dir, f"{group_name}_interaction_regression.txt")
        with open(txt_path, 'w') as f:
            f.write(str(interaction_result.summary()))
        print(f"Saved -> {txt_path}")
        saved_paths['interaction_regression'] = txt_path

    return saved_paths


def save_all_layer_analysis_outputs(output_dir, layer_analysis_results):
    """
    Loop save_layer_analysis_group_outputs over every group.

    Parameters
    ----------
    output_dir : str
    layer_analysis_results : dict
        {group_name: run_layer_analysis_for_group(...) result}.

    Returns
    -------
    saved_paths_by_group : dict
    """
    saved_paths_by_group = {}
    for group_name, result in layer_analysis_results.items():
        saved_paths_by_group[group_name] = save_layer_analysis_group_outputs(output_dir, group_name, result)
    print(f"\nSaved layer-analysis outputs for {len(saved_paths_by_group)} group(s) to {output_dir}")
    return saved_paths_by_group


In [64]:
# --- Save everything Phase 5 generated for this animal ---
PHASE5_OUTPUT_DIR = os.path.join(TEST_ANIMAL_DIR, 'Phase5_LayerSpecific_Results')

saved_paths = save_all_layer_analysis_outputs(PHASE5_OUTPUT_DIR, layer_analysis_results)


Saved -> D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD\Phase5_LayerSpecific_Results\Active_OL_layer_sample_sizes.csv
Saved -> D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD\Phase5_LayerSpecific_Results\Active_OL_depth_sample_sizes.csv
Saved -> D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD\Phase5_LayerSpecific_Results\Active_OL_layer_pairwise_stats.csv
Saved -> D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD\Phase5_LayerSpecific_Results\Active_OL_depth_pairwise_stats.csv
Saved -> D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD\Phase5_LayerSpecific_Results\Active_OL_layer_omnibus_stats.json
Saved -> D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD\Phase5_LayerSpecific_Results\Active_OL_depth_omnibus_stats.json
Saved -> D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD\Phase5_LayerSpecific_Results\Active_OL_layer_comparison_plot.png
Saved -> D:\V1_SpatialModulation\2p\V

## Function 5.15 -- pool DCZ1/DCZ2/DCZ3 and compare by layer

Combines baseline/saline/dcz SMI across the three closed-loop groups (DCZ1/DCZ2/DCZ3) -- every cell in every session counted individually, not summarized to a per-session value first. Reuses Function 5.3's `plot_smi_by_layer` design (it's already fully generic over which table you feed it), just under a named wrapper for this specific pooled-across-groups view.

DCZ1/DCZ2/DCZ3's tables each carry their own copy of the shared baseline (Day5) session -- naive concatenation would triple-count every baseline cell. Deduplicated on `(session_label, cell_idx)`, same fix Phase 6's `build_all_sessions_landmark_smi_table` already applies to this exact problem.

In [65]:
def build_all_dcz_pooled_table(all_group_dfs, groups=('DCZ1', 'DCZ2', 'DCZ3')):
    """
    Pool the given comparison groups' per-cell tables into one table --
    every cell in every session counted individually (not averaged within
    a session first). Defaults to the three closed-loop groups (DCZ1/
    DCZ2/DCZ3), excluding Active_OL/Stationary_OL since those are a
    different manipulation (open loop).

    The groups' tables each carry their OWN copy of the shared baseline
    (Day5) session -- concatenating naively would triple-count every
    baseline cell. Deduplicated on (session_label, cell_idx), same fix
    Phase 6's build_all_sessions_landmark_smi_table already applies to
    this exact problem.

    Parameters
    ----------
    all_group_dfs : dict
        {group_name: df}, from load_all_group_dfs_from_phase4. Must
        contain every name in `groups`.
    groups : tuple of str

    Returns
    -------
    pooled_df : pandas.DataFrame
        One row per unique (session_label, cell_idx) across the pooled
        groups.
    """
    missing = [g for g in groups if g not in all_group_dfs]
    if missing:
        raise KeyError(f"{missing} not in all_group_dfs -- available: {list(all_group_dfs.keys())}")

    pooled_df = pd.concat([all_group_dfs[g] for g in groups], ignore_index=True)
    n_before = len(pooled_df)
    pooled_df = pooled_df.drop_duplicates(subset=['session_label', 'cell_idx']).reset_index(drop=True)
    n_after = len(pooled_df)

    print(f"Pooled {groups}: {n_before} cell-rows -> {n_after} after dropping duplicate "
          f"(session_label, cell_idx) rows (shared baseline counted once, not {len(groups)}x).")
    print("\nSessions contributing per condition:")
    print(pooled_df.groupby('condition')['session_label'].nunique().rename('n_sessions').to_string())
    print("\nCell-rows per condition:")
    print(pooled_df.groupby('condition').size().rename('n_cell_rows').to_string())

    return pooled_df


def test_all_dcz_by_layer_paired(all_group_dfs, groups=('DCZ1', 'DCZ2', 'DCZ3'),
                                  layer_col='layer', group_col='condition', value_col='SMI',
                                  filter_col='valid', summary_stat='median'):
    """
    Per-layer saline-vs-dcz significance, done the way this project
    actually validated it -- NOT the unpaired cell-pooled t-test tried
    first (scrapped: pools hundreds of cells from one session as if they
    were independent replicates, the same pseudo-replication problem
    already caught and fixed in this phase's Functions 5.11-5.13 and
    discarded outright in Phase 6). Each of `groups` (DCZ1/DCZ2/DCZ3 by
    default) is ONE independent pair; a paired t-test (+ Wilcoxon) runs
    across those n=len(groups) pairs. Same design as Function 5.12's
    test_paired_smi_significance_by_layer (and the ad hoc closed-loop-only
    version computed earlier in this conversation), reimplemented directly
    from all_group_dfs since this notebook doesn't carry Functions
    5.11-5.13 (added straight to the .py per an earlier explicit
    instruction).

    Only saline vs dcz is tested here (not baseline) -- matches "closed
    loop saline vs dcz sessions" exactly as asked, even though the pooled
    plot itself still shows baseline alongside for visual context.

    Parameters
    ----------
    all_group_dfs : dict
        {group_name: df}, from load_all_group_dfs_from_phase4.
    groups : tuple of str
        Which comparison groups count as the independent pairs.
    layer_col, group_col, value_col, filter_col : str
    summary_stat : str
        'median' or 'mean' -- the per-group-per-layer summary computed
        before pairing.

    Returns
    -------
    stats_by_layer : dict
        {layer: {'n_pairs', 'group_values' (DataFrame: group, saline, dcz,
        n_saline, n_dcz), 'mean_diff', 'paired_t_stat', 'paired_t_p',
        'wilcoxon_stat', 'wilcoxon_p'} or None if fewer than 2 of `groups`
        have both saline and dcz cells present for that layer}.
    """
    stat_fn = np.median if summary_stat == 'median' else np.mean

    missing = [g for g in groups if g not in all_group_dfs]
    if missing:
        raise KeyError(f"{missing} not in all_group_dfs -- available: {list(all_group_dfs.keys())}")

    layers_present = sorted({l for g in groups for l in all_group_dfs[g][layer_col].dropna().unique()})
    layer_order = _layer_order(layers_present)

    stats_by_layer = {}
    for layer in layer_order:
        rows = []
        for g in groups:
            df = all_group_dfs[g]
            layer_df = df[df[filter_col] & (df[layer_col] == layer)]
            saline_vals = layer_df.loc[layer_df[group_col] == 'saline', value_col].to_numpy()
            dcz_vals = layer_df.loc[layer_df[group_col] == 'dcz', value_col].to_numpy()
            if len(saline_vals) == 0 or len(dcz_vals) == 0:
                continue
            rows.append({
                'group': g,
                'saline': float(stat_fn(saline_vals)), 'dcz': float(stat_fn(dcz_vals)),
                'n_saline': len(saline_vals), 'n_dcz': len(dcz_vals),
            })

        group_values = pd.DataFrame(rows)
        print(f"\n--- Layer {layer} ---")
        if len(group_values) < 2:
            print(f"  only {len(group_values)}/{len(groups)} group(s) have both saline and dcz cells "
                  f"-- skipping (need at least 2 pairs).")
            stats_by_layer[layer] = None
            continue

        print(group_values.to_string(index=False))
        diffs = group_values['saline'] - group_values['dcz']
        t_stat, t_p = ttest_rel(group_values['saline'], group_values['dcz'])
        w_stat, w_p = wilcoxon(group_values['saline'], group_values['dcz'])

        stats_by_layer[layer] = {
            'n_pairs': len(group_values), 'group_values': group_values,
            'mean_diff': float(diffs.mean()),
            'paired_t_stat': float(t_stat), 'paired_t_p': float(t_p),
            'wilcoxon_stat': float(w_stat), 'wilcoxon_p': float(w_p),
        }
        print(f"  mean diff (saline - dcz): {diffs.mean():.4f}")
        print(f"  paired t-test: stat={t_stat:.3f}, p={t_p:.4f}")
        print(f"  wilcoxon:      stat={w_stat:.3f}, p={w_p:.4f}")

    return stats_by_layer


def _sig_stars(p):
    """'***'/'**'/'*' for p < 0.001/0.01/0.05, else None."""
    if p < 0.001:
        return '***'
    if p < 0.01:
        return '**'
    if p < 0.05:
        return '*'
    return None


def plot_all_dcz_layer_comparison(pooled_df, stats_by_layer=None,
                                   title='All DCZ (DCZ1+DCZ2+DCZ3 pooled): SMI by layer and condition',
                                   layer_col='layer', group_col='condition', value_col='SMI',
                                   filter_col='valid'):
    """
    Per-layer violin+strip plot of every individual pooled cell (same
    visual design as Function 5.3's plot_smi_by_layer) -- with ONE
    significance bracket per layer (saline vs dcz), drawn from
    stats_by_layer's PAIRED t-test across DCZ1/DCZ2/DCZ3
    (test_all_dcz_by_layer_paired), NOT from any test computed on the
    pooled cells themselves. The plot stays cell-level (every individual
    cell shown, for the visual distribution); the significance annotation
    is deliberately computed at the correct independent-unit level (n=3
    groups) and overlaid on top -- these are two different things, kept
    intentionally separate so the stars reflect a real, defensible test
    rather than the pseudo-replicated cell-pooled version that was tried
    and scrapped first. Reimplemented rather than calling plot_smi_by_layer
    directly, since that function is shared with every other group's plot.
    Save as 'all_dcz_layer_comparison_plot.svg' (vector, not raster --
    per-request for this figure specifically).

    Parameters
    ----------
    pooled_df : pandas.DataFrame
        From build_all_dcz_pooled_table (individual cells, for the violins).
    stats_by_layer : dict or None
        {layer: {..., 'paired_t_p', ...} or None}, from
        test_all_dcz_by_layer_paired. Pass None to skip annotation.
    title : str
    layer_col, group_col, value_col, filter_col : str

    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    filtered = pooled_df[pooled_df[filter_col]]
    layers_present = [l for l in filtered[layer_col].dropna().unique()]
    layer_order = _layer_order(layers_present)

    color_by_category = {'baseline': 'tab:blue', 'saline': 'tab:orange', 'dcz': 'tab:green'}

    n_layers = len(layer_order)
    n_cols = 2
    n_rows = int(np.ceil(n_layers / n_cols)) if n_layers > 0 else 1
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(8 * n_cols, 5 * n_rows))
    axes = np.atleast_1d(axes).flatten()

    for ax, layer in zip(axes, layer_order):
        layer_df = filtered[filtered[layer_col] == layer]
        category_order = [c for c in ['baseline', 'saline', 'dcz'] if c in layer_df[group_col].unique()]
        category_order += [c for c in layer_df[group_col].unique() if c not in category_order]

        data_by_category = [layer_df.loc[layer_df[group_col] == cat, value_col].to_numpy()
                            for cat in category_order]

        if len(data_by_category) == 0 or all(len(d) == 0 for d in data_by_category):
            ax.set_title(f"{layer} (no data)")
            ax.axis('off')
            continue

        parts = ax.violinplot(data_by_category, showmedians=True)
        for i, body in enumerate(parts['bodies']):
            body.set_facecolor(color_by_category.get(category_order[i], 'gray'))
            body.set_alpha(0.4)

        rng = np.random.default_rng(0)
        for i, vals in enumerate(data_by_category):
            jitter = rng.uniform(-0.08, 0.08, size=len(vals))
            ax.scatter(np.full(len(vals), i + 1) + jitter, vals,
                       color=color_by_category.get(category_order[i], 'gray'), s=12, alpha=0.5)

        ax.set_xticks(range(1, len(category_order) + 1))
        ax.set_xticklabels(category_order)
        ax.set_ylabel(value_col)
        ax.set_title(layer)
        ax.axhline(0, color='gray', linestyle='--', alpha=0.5)

        # --- Significance bracket: saline vs dcz, from the paired
        # (DCZ1/DCZ2/DCZ3-as-independent-pairs) t-test, NOT from the cells
        # shown in this panel. ---
        data_max = max((v.max() for v in data_by_category if len(v) > 0), default=1.0)
        data_min = min((v.min() for v in data_by_category if len(v) > 0), default=-1.0)
        bracket_y = data_max + 0.08
        drew_bracket = False

        layer_stat = stats_by_layer.get(layer) if stats_by_layer is not None else None
        if layer_stat is not None:
            stars = _sig_stars(layer_stat['paired_t_p'])
            x_pos = {cat: i + 1 for i, cat in enumerate(category_order)}
            if stars is not None and 'saline' in x_pos and 'dcz' in x_pos:
                x1, x2 = x_pos['saline'], x_pos['dcz']
                ax.plot([x1, x1, x2, x2], [bracket_y, bracket_y + 0.025, bracket_y + 0.025, bracket_y],
                        color='black', linewidth=1.2)
                ax.text((x1 + x2) / 2, bracket_y + 0.03, stars, ha='center', va='bottom', fontsize=22)
                drew_bracket = True

        top = bracket_y + 0.15 if drew_bracket else max(data_max + 0.15, 1.05)
        ax.set_ylim(min(data_min - 0.15, -1.05), top)

    for ax in axes[len(layer_order):]:
        ax.axis('off')

    fig.suptitle(title, fontsize=20, fontweight='bold')
    plt.tight_layout()
    return fig

In [66]:
# --- Pool DCZ1/DCZ2/DCZ3 (closed-loop only): plot every individual cell, test saline-vs-dcz
# properly (paired across the 3 groups as independent units) ---
all_dcz_pooled_df = build_all_dcz_pooled_table(all_group_dfs)
all_dcz_stats_by_layer = test_all_dcz_by_layer_paired(all_group_dfs)
all_dcz_layer_comparison_fig = plot_all_dcz_layer_comparison(all_dcz_pooled_df, stats_by_layer=all_dcz_stats_by_layer)
plt.show()

os.makedirs(PHASE5_OUTPUT_DIR, exist_ok=True)
all_dcz_svg_path = os.path.join(PHASE5_OUTPUT_DIR, 'all_dcz_layer_comparison_plot.svg')
all_dcz_layer_comparison_fig.savefig(all_dcz_svg_path, bbox_inches='tight')
print(f"Saved -> {all_dcz_svg_path}")

Pooled ('DCZ1', 'DCZ2', 'DCZ3'): 3834 cell-rows -> 3834 after dropping duplicate (session_label, cell_idx) rows (shared baseline counted once, not 3x).

Sessions contributing per condition:
condition
dcz       3
saline    3

Cell-rows per condition:
condition
dcz       2135
saline    1699

--- Layer L2/3 ---
group   saline      dcz  n_saline  n_dcz
 DCZ1 0.667739 0.580937         3     24
 DCZ2 0.756741 0.671867         9     28
 DCZ3 0.734185 0.699328         4     33
  mean diff (saline - dcz): 0.0688
  paired t-test: stat=4.049, p=0.0559
  wilcoxon:      stat=0.000, p=0.2500

--- Layer L4 ---
group   saline      dcz  n_saline  n_dcz
 DCZ1 0.808577 0.605647         8     42
 DCZ2 0.701425 0.689597        15     54
 DCZ3 0.866134 0.812824        11     49
  mean diff (saline - dcz): 0.0894
  paired t-test: stat=1.540, p=0.2635
  wilcoxon:      stat=0.000, p=0.2500

--- Layer L5 ---
group   saline      dcz  n_saline  n_dcz
 DCZ1 0.640999 0.527858        22     56
 DCZ2 0.831529 0.67190